# 04 — Tổng hợp kết quả (MILK10k)

Gộp output của các notebook 01, 02, 03 thành bảng báo cáo: baseline backbone × nhánh ảnh, phương pháp imbalance ×
nhánh ảnh, mô hình ML, top run, và chi tiết theo lớp của run tốt nhất. Không cần GPU.

**Chuẩn bị:** Add Input → Your Work → Notebooks → thêm output của 01, 02, 03, rồi điền các thư mục `outputs` vào `RESTORE`.

In [ ]:
# ===================== CẤU HÌNH =====================
REPO_URL = "https://github.com/trong5nhan6/fussion.git"
BRANCH = "main"
DRIVE_URL = "https://drive.google.com/file/d/1lM1nB_4pq3Vg-GwlaTMQjtus_bpbMhGX/view?usp=drive_link"  # zip dữ liệu MILK10k
DATA_SOURCE = None    # hoặc đường dẫn Kaggle Dataset, vd "/kaggle/input/milk10k" (khi đó không tải từ Drive)

# Chạy tiếp / dùng lại kết quả: gắn output của notebook trước làm Input (Add Input -> Your Work -> Notebooks),
# rồi liệt kê thư mục outputs của nó, vd "/kaggle/input/<ten-notebook>/fussion/outputs"
RESTORE = []

## 1. Cài đặt

In [ ]:
import os, glob
from IPython.display import Markdown, display

REPO_DIR = "/kaggle/working/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
DATA_DIR = "/tmp/MILK10k"               # ngoài /kaggle/working -> dữ liệu không bị lưu vào output notebook
os.environ["MILK10K_ROOT"] = DATA_DIR   # code tự đọc biến này thay cho data.root trong config

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q gdown
!git log -1 --format="Commit: %h %s"

## 2. Bảng tổng hợp

In [ ]:
for r in RESTORE:
    print(r, "->", "OK" if os.path.isdir(r) else "KHÔNG TỒN TẠI")
dirs = " ".join(RESTORE) or "outputs"
!python scripts/summarize.py --dirs {dirs} --out outputs/summary > /dev/null
display(Markdown(open("outputs/summary/summary.md", encoding="utf-8").read()))

## 3. Run tốt nhất: kết quả theo từng lớp

In [ ]:
import json, shutil
import pandas as pd

runs = pd.read_csv("outputs/summary/all_runs.csv").sort_values("Dice Coefficient", ascending=False)
best = runs.iloc[0]
print(f"Run tốt nhất: {best.run} | Dice = {best['Dice Coefficient']:.4f}")
run_dir = next(os.path.join(d, best.run) for d in (RESTORE or ["outputs"]) if os.path.isdir(os.path.join(d, best.run)))
m = json.load(open(os.path.join(run_dir, "metrics.json"), encoding="utf-8"))
per_class = pd.DataFrame(m["overall"]["per_class"]).T[["n", "auc", "ap", "accuracy", "sensitivity", "specificity", "dice"]]
per_class.columns = ["n", "AUC", "Average Precision", "Accuracy", "Sensitivity", "Specificity", "Dice Coefficient"]
per_class.loc["Macro average"] = [per_class.n.sum()] + [m["overall"][k] for k in ["auc", "ap", "accuracy", "sensitivity", "specificity", "dice"]]
display(per_class.style.format({c: "{:.4f}" for c in per_class.columns if c != "n"}))

shutil.copy(os.path.join(run_dir, "submission.csv"), "/kaggle/working/best_submission.csv")
print("File nộp của run tốt nhất: /kaggle/working/best_submission.csv")

## 4. Learning curve của run ảnh tốt nhất

In [ ]:
import matplotlib.pyplot as plt

img_runs = runs[runs.kind == "image"]
if img_runs.empty:
    raise SystemExit("Chưa có run ảnh nào (chỉ có mô hình ML) -> không có learning curve")
best_img = img_runs.iloc[0]
img_dir = next(os.path.join(d, best_img.run) for d in (RESTORE or ["outputs"]) if os.path.isdir(os.path.join(d, best_img.run)))
h = pd.read_csv(os.path.join(img_dir, "fold0", "history.csv"))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(h.epoch, h.train_loss, label="train loss", lw=2)
axes[0].plot(h.epoch, h.val_loss, label="val loss (CE)", lw=2)
axes[0].set_xlabel("epoch"); axes[0].set_title("Loss"); axes[0].legend(frameon=False)
for k, name in [("dice", "Dice"), ("auc", "AUC"), ("sensitivity", "Sensitivity")]:
    axes[1].plot(h.epoch, h[k], label=name, lw=2)
axes[1].set_xlabel("epoch"); axes[1].set_title("Metric val (macro)"); axes[1].legend(frameon=False)
for ax in axes:
    ax.grid(alpha=0.3); ax.spines[["top", "right"]].set_visible(False)
plt.suptitle(best_img.run); plt.tight_layout(); plt.show()